# Chat history endpoints -- live test

Tests `GET /conversations` and `GET /conversations/{id}/history` directly
against `app/gateway/gateway.py`, same pattern as `phase3_gateway_e2e.ipynb`:
`httpx.AsyncClient` over `ASGITransport`, no uvicorn, no deployed service.
Scoped to just these two read endpoints -- `/ask`, `/ask/status`, and
`/ask/cancel` are already covered there.

Both endpoints read `agent_chat_history.turns` via `turn-writer-sa`
impersonation, so this needs the same local impersonation grant already
set up for the writer-side tests (`local-dev-environment-setup.md`).

No MCP server needed -- neither endpoint touches the orchestrator.

In [10]:
import sys
sys.path.insert(0, "..")

import httpx

import app.gateway.gateway as gateway
from app.gateway.models import ConversationHistoryResponse, ConversationListResponse

# Edit this to a real claims["oid"] that already has rows in agent_chat_history.turns --
# run a turn or two through phase3_gateway_e2e.ipynb first if you need fresh data.
USER_ID = "gateway-test-user"
#notebook-test-user
#gateway-test-user
#17d6e75b-3a37-496e-a310-b8095ac658b6

# Stand-ins for the Entra token and API-key checks; every route runs for real.
gateway.validate_api_key = lambda x_api_key: None
gateway.validate_entra_token = lambda authorization: {"oid": USER_ID}

client = httpx.AsyncClient(transport=httpx.ASGITransport(app=gateway.app), base_url="http://gateway")
HEADERS = {"Authorization": "Bearer stand-in", "x-api-key": "stand-in"}

## GET /conversations

Lists `USER_ID`'s past conversations, most recent first.

In [11]:
resp = await client.get("/conversations", headers=HEADERS)
resp.raise_for_status()
conversations = ConversationListResponse(**resp.json()).conversations

for c in conversations:
    print(f"{c.last_activity_at}  {c.conversation_id}  {c.title!r}")

2026-10-14 20:37:54.483414+00:00  fa76b3e9-a5df-4067-b540-807bbade71b4  'Show me a heatmap of order volume by day of week and hour o...'
2026-10-14 04:03:07.671360+00:00  473c24e8-05ee-4064-b1ab-3cdd0a790fec  'How many unique products are there? How many unique users?...'
2026-10-14 03:57:51.843772+00:00  e3029d64-f811-49ac-a72b-30f31664ff58  'Show me a concentration chart of product order volume  what...'


## GET /conversations/{id}/history

Defaults to the most recent conversation from the list above -- edit
`CONVERSATION_ID` directly to target a specific one instead.

In [12]:
CONVERSATION_ID = conversations[0].conversation_id if conversations else ""
print("CONVERSATION_ID:", CONVERSATION_ID)

CONVERSATION_ID: fa76b3e9-a5df-4067-b540-807bbade71b4


In [13]:
resp = await client.get(f"/conversations/{CONVERSATION_ID}/history", headers=HEADERS)
resp.raise_for_status()
turns = ConversationHistoryResponse(**resp.json()).turns

for i, t in enumerate(turns):
    print(f"--- turn {i} ---")
    print("question:", t.question)
    print("sources:", t.sources)
    print("suggested_follow_ups:", t.suggested_follow_ups)
    print("needs_approval:", t.needs_approval)
    print()

--- turn 0 ---
question: Show me a heatmap of order volume by day of week and hour of day.
sources: ['Query warehouse', 'Create chart']
suggested_follow_ups: ['Which departments drive the Sunday/Monday order surge?', 'How does days_since_prior_order relate to these peak ordering times?', "What's the reorder rate during peak vs. off-peak hours?"]
needs_approval: False

--- turn 1 ---
question: How many unique products are there? How many unique users? How many unique orders?
sources: ['Query warehouse']
suggested_follow_ups: ["What's the average number of orders per user?", 'Which products are ordered most frequently?', "What's the average basket size across all orders?"]
needs_approval: False



In [16]:
from IPython.display import HTML, display

# Rendered view of one turn's answer_html -- same control the Power Apps
# HTML text control binds to.
TURN_INDEX = 0
display(HTML(turns[TURN_INDEX].answer_html))

## Ownership check -- a different user can't read this conversation's history

`GET /conversations/{id}/history` takes a caller-supplied `conversation_id`,
so it runs `assert_owns_conversation` first (unlike `/conversations`, which
has no `conversation_id` to check). Confirms it actually returns 404, not
just that it has the code path.

In [17]:
gateway.validate_entra_token = lambda authorization: {"oid": "someone-else"}

resp = await client.get(f"/conversations/{CONVERSATION_ID}/history", headers=HEADERS)
print(resp.status_code, resp.json())
assert resp.status_code == 404

# Restore USER_ID for any further cells in this notebook.
gateway.validate_entra_token = lambda authorization: {"oid": USER_ID}

404 {'detail': 'Conversation not found.'}
